# مشروع الدبلجة العربية — Course Translation V2.6.1 + Generic Course Pipeline V2.8 (Colab)

**الهدف الآن:** الاستفادة من الـ17 ترجمة الموجودة، ثم فحص الجودة والصوت مجاناً عند اختيارك، بدون إعادة مراحل WhisperX أو الاعتماد النهائي آلياً.

**اختيارات التشغيل داخل الأقسام 6 و7:** `reuse_existing`، `review_existing`، `regenerate_selected` للترجمة؛ و`plan_only`، `edge_online`، `mms_local`، `gemini_flash_lite` للصوت.

**الأمان:** كل اتصال خارجي أو توليد صوت يحتاج `RUN_... = True` وإذن مستقل. **لا تُشغّل خلية الاختيار الافتراضي أي API أو TTS**. احتفظ بـMaster الأصلي والـIDs وتوقيت الكلمات، ولا تملأ `final_dub_ar`.


## 1 — ربط Drive وتحديث الكود الحالي من GitHub دون حذف التغييرات

عند وجود نسخة GitHub داخل جلسة Colab، نجري `git pull --ff-only` فقط إذا كانت نظيفة. لا يُمس أي ملف بيانات في Drive، ولا يعاد WhisperX. إذا كان checkout معدلاً محلياً يوقف التحديث حتى لا يمحو تغييراتك.


In [ ]:
from pathlib import Path
from google.colab import drive
import json, hashlib, sys, subprocess

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ArabicDubbingV2')
OUTPUTS = PROJECT / 'outputs'
OUTPUTS.mkdir(parents=True, exist_ok=True)

# المصدر الوحيد للكود هو GitHub؛ لا حاجة لرفع ZIP إلى Drive.
REPO_URL = 'https://github.com/HamzaAlSamman/AI-Dubbing-Project.git'
REPO_BRANCH = 'main'
CODE_DIR = Path('/content/AI-Dubbing-Project')
if CODE_DIR.exists():
    if not (CODE_DIR / '.git').is_dir():
        raise FileExistsError(f'Folder exists but is not a Git repository: {CODE_DIR}')
    changed = subprocess.check_output(['git','-C',str(CODE_DIR),'status','--porcelain'],text=True).strip()
    if changed:
        raise RuntimeError('Colab GitHub checkout has uncommitted changes. Save them before refreshing code. No reset performed.')
    subprocess.run(['git','-C',str(CODE_DIR),'pull','--ff-only','origin',REPO_BRANCH],check=True)
    print('Updated existing clean GitHub checkout safely:', CODE_DIR)
else:
    subprocess.run(['git','clone','--depth','1','--branch',REPO_BRANCH,REPO_URL,str(CODE_DIR)],check=True)

if not (CODE_DIR / 'arabic_dubbing_v2' / 'workflow.py').is_file():
    raise FileNotFoundError('GitHub checkout is incomplete: arabic_dubbing_v2/workflow.py missing')
commit=subprocess.check_output(['git','-C',str(CODE_DIR),'rev-parse','HEAD'],text=True).strip()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print('GitHub source commit:', commit)
print('No ASR, translation or TTS executed.')


## 2 — اختبار الكود دون تشغيل النموذج أو استهلاك GPU

In [ ]:
checks = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                         cwd=CODE_DIR, capture_output=True, text=True)
print(checks.stdout)
print(checks.stderr[-10000:])
if checks.returncode != 0:raise RuntimeError('Failed offline tests; stop before processing any project Master.')
print('Offline tests passed. This is not a substitute for real course validation.')

## 3 — اختيار أحدث Master الحقيقي من Drive

النسخة `V2.5` المؤكدة محفوظة عادةً في `outputs/lesson/gemini_translation_v1/`. لا ننتقل إلى إصدار أقدم تلقائياً. عند تعدد المسارات أضبط `EXPLICIT_MASTER_PATH` صراحةً.


In [ ]:
# اسم مسودة الترجمة الحالية؛ هذا ليس ملفاً معتمداً للدبلجة.
MASTER_FILENAME = 'translation_master_v2_5_editorial_candidate.json'
RUN_DIR = OUTPUTS / 'lesson'
EXPLICIT_MASTER_PATH = None

# تعريف بيانات الكورس اختياري، دون قاموس مخصص لقسم علمي معين.
COURSE_METADATA_OVERRIDES = {
    'id': None, 'title': None, 'description': None,
    'chapter': None, 'lesson': None, 'topic': None,
}

available = sorted(OUTPUTS.rglob('translation_master*.json'))
print('Masters discovered in Drive:')
for f in available[:80]:
    print(' -', f)
if EXPLICIT_MASTER_PATH:
    MASTER_PATH = Path(EXPLICIT_MASTER_PATH)
else:
    preferred = RUN_DIR / 'gemini_translation_v1' / MASTER_FILENAME
    fallback = RUN_DIR / MASTER_FILENAME
    if preferred.is_file():
        MASTER_PATH = preferred
    elif fallback.is_file():
        MASTER_PATH = fallback
    else:
        matches = [f for f in available if f.name == MASTER_FILENAME]
        if len(matches) != 1:
            raise FileNotFoundError(
                f'Expected exactly one {MASTER_FILENAME}; found {len(matches)}. '
                'Set EXPLICIT_MASTER_PATH to the reviewed course master.'
            )
        MASTER_PATH = matches[0]
if not MASTER_PATH.is_file():
    raise FileNotFoundError('Specified Master not found: ' + str(MASTER_PATH))
print('Selected master:', MASTER_PATH)
print('SHA256:', hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest())


## 4 — Data Contract + Context Builder + Dynamic Glossary + Educational QA

يستخرج مؤشرات المجال من بيانات الكورس أو مصطلحات متكررة فيه، ثم يحفظها **كمرشحات غير معتمدة**. جميع فحوص المعنى غير المنجزة تبقى `unknown`، ولا يتغير المصدر.

In [ ]:
from arabic_dubbing_v2.workflow import prepare_project
from arabic_dubbing_v2.common import fingerprint

src_bytes=MASTER_PATH.read_bytes()
src_hash=hashlib.sha256(src_bytes).hexdigest()
# نسخة جديدة باسم مرتبط ببصمة المصدر؛ لا تُستبدل إن كانت موجودة بالفعل.
OUT_DIR=RUN_DIR/'translation_v2_figma_implementation'
OUT_DIR.mkdir(parents=True,exist_ok=True)
DERIVED_MASTER = OUT_DIR / f'translation_master_v2_6_derived_{src_hash[:12]}.json'
from arabic_dubbing_v2.common import read_json
if DERIVED_MASTER.exists():
    prepared = read_json(DERIVED_MASTER)
    if prepared['translation_v2_metadata'].get('source_master_sha256') != src_hash:
        raise RuntimeError('Derived file belongs to a different source: ' + str(DERIVED_MASTER))
    print('Reusing verified derived Master without overwriting:', DERIVED_MASTER)
else:
    prepared = prepare_project(MASTER_PATH,DERIVED_MASTER,course_overrides=COURSE_METADATA_OVERRIDES)
assert hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest() == src_hash
assert all((u.get('analysis_v2') or {}).get('review_gate')=='pending_human_bilingual_review'
           for u in prepared['translation_units'])
print('Master derived:',DERIVED_MASTER)
print('Report:',json.dumps(prepared['translation_v2_report'],ensure_ascii=False,indent=2)[:12000])
print('Domain hint:',prepared['terminology_v2']['domain_analysis'])
print('Candidate terms:',len(prepared['terminology_v2']['candidates']))
print('No model calls; original translation texts, ASR, word refs and timestamps remain unchanged.')

## 5 — استخراج ملفات خطط ترجمة ومراجعة المصطلحات (لا LLM هنا)

يدعم هذا القسم أربع مراحل Prompt منفصلة: accurate → natural → timing_candidate → semantic QA، ويضيف خطط Contextual Validation للمصطلحات المكتشفة. **لن ينفّذ هذه المطالبات أو يستهلك API.**

In [ ]:
from arabic_dubbing_v2.common import save_new_json
from arabic_dubbing_v2.translation import build_all_pass_plans
from arabic_dubbing_v2.terminology import validation_plans

PLAN_TAG = src_hash[:12]
TRANSLATION_PLANS=OUT_DIR / f'translation_pass_plans_{PLAN_TAG}.json'
GLOSSARY_PLANS=OUT_DIR / f'dynamic_glossary_validation_plans_{PLAN_TAG}.json'
translation_plan=build_all_pass_plans(prepared)
glossary_plan=validation_plans(prepared['terminology_v2'],prepared['translation_v2_metadata']['course'])
from arabic_dubbing_v2.common import read_json
for path, value in [(TRANSLATION_PLANS,translation_plan),(GLOSSARY_PLANS,glossary_plan)]:
    if path.exists():
        if read_json(path) != value:
            raise RuntimeError('Existing saved plan differs; refusing overwrite: ' + str(path))
        print('Existing plan verified and reused:', path)
    else:
        save_new_json(path,value)
print('TU plans:',len(translation_plan['plans']), '| Dynamic glossary plans:',len(glossary_plan['plans']))
print('Saved:',TRANSLATION_PLANS,GLOSSARY_PLANS)
print('NO LLM REQUESTS WERE MADE.')

## 5A — تعميم الكورس والقاموس الديناميكي V2.7 (بدون API)

هذه خطوة مستقلة تعمل بعد تحميل الكود وتحديد Master في القسم 3. **لا تعِد تشغيل مراحل V2.6 التي أنتجت ملفات بالفعل.** تميّز الهوية التجريبية، تقترح المجال مع دليل، وتعدّ قائمة مرشحات المصطلحات وفحص الاتساق فقط عند وجود موافقات صريحة، وتحفظ تقريراً جديداً دون تغيير المصدر أو التوقيتات. لا تؤكد المعنى آلياً.


In [ ]:
# 5A — Generic Course Glossary V2.7; offline and idempotent.
# Run after sections 1 and 3; no need to regenerate existing V2.6 master/plans.
from arabic_dubbing_v2.workflow import enrich_master
from arabic_dubbing_v2.course_glossary_v27 import build_course_glossary_report
from arabic_dubbing_v2.common import save_new_json, read_json
import json, hashlib
from pathlib import Path

GENERIC_GLOSSARY_V27 = True
if GENERIC_GLOSSARY_V27:
    if not Path(MASTER_PATH).is_file():
        raise FileNotFoundError('Select an existing Master in section 3 first')
    source_hash_v27 = hashlib.sha256(Path(MASTER_PATH).read_bytes()).hexdigest()
    raw_v27 = read_json(MASTER_PATH)
    prepared_v27 = enrich_master(raw_v27, course_overrides=COURSE_METADATA_OVERRIDES,
        source_filename=str(MASTER_PATH), source_sha256=source_hash_v27)
    report_v27 = build_course_glossary_report(prepared_v27)
    generic_dir = RUN_DIR / 'translation_v2_figma_implementation'
    generic_dir.mkdir(parents=True, exist_ok=True)
    generic_path = generic_dir / f'course_glossary_v2_7_{source_hash_v27[:12]}.json'
    if generic_path.is_file():
        old_v27 = read_json(generic_path)
        if old_v27['source_master_data_fingerprint'] != report_v27['source_master_data_fingerprint']:
            raise RuntimeError('Existing V2.7 report belongs to a different source; stop')
        print('Reused existing V2.7 report:', generic_path)
    else:
        save_new_json(generic_path, report_v27)
        print('Saved new V2.7 report:', generic_path)
    if hashlib.sha256(Path(MASTER_PATH).read_bytes()).hexdigest() != source_hash_v27:
        raise RuntimeError('Source master changed during report generation')
    print('Course identity:', json.dumps(report_v27['course_identity'], ensure_ascii=False))
    print('Domain evidence:', json.dumps(report_v27['domain_detection'], ensure_ascii=False))
    print('Term candidates:', report_v27['candidate_count'], 'approved:', len(report_v27['approved_memory']))
    print('Sample terms:', [x['source_term'] for x in report_v27['candidates'][:12]])
    print('Source immutable; no LLM, API, ASR or TTS executed.')

## 5B — Multi-Lesson Registry + Contextual Terminology Review V2.8 (بدون API)

يقرأ Master الموجود فقط. لا تغيّر `CANONICAL_COURSE=None` إلى هوية مؤكدة حتى تملك العنوان والمعرّف الحقيقيين. يستخدم `lesson_id/tu_id` لمنع تصادم معرّفات الوحدات، ويصدر قائمة مراجعة المصطلحات وذاكرة معتمدة فارغة افتراضياً. لا يعيد تشغيل مراحل سابقة ولا يعتمد ترجمة أو صوتاً.


In [ ]:
# 5B — Multi-lesson course registry + contextual terminology queue (V2.8).
# Offline, does not generate translations, approve terminology or run TTS/API.
from arabic_dubbing_v2.workflow import enrich_master
from arabic_dubbing_v2.course_pipeline_v28 import build_pipeline_snapshot
from arabic_dubbing_v2.common import read_json, save_new_json, fingerprint
from pathlib import Path
import hashlib, json

# Replace ONLY with details confirmed by the course owner. Never guess from the pilot transcript.
CANONICAL_COURSE = None
# Example AFTER verification:
# CANONICAL_COURSE = {'id':'actual_course_id', 'title':'Verified course title', 'confirmed_by':'course_owner'}
CURRENT_LESSON_ID = 'pilot_lesson_01'
EXTRA_LESSON_MASTERS = []  # Optional: [('lesson_02', '/content/drive/MyDrive/.../master.json')]
EXPLICIT_APPROVED_TERMS = []  # Only audited human-approved entries, NOT machine suggestions.

selected = [(CURRENT_LESSON_ID, str(MASTER_PATH))] + list(EXTRA_LESSON_MASTERS)
lesson_list=[]; file_checks=[]
for lesson_id, path in selected:
    master_file=Path(path)
    if not master_file.is_file():raise FileNotFoundError(master_file)
    start_sha=hashlib.sha256(master_file.read_bytes()).hexdigest()
    prepared_for_lesson=enrich_master(read_json(master_file),
        source_filename=str(master_file),source_sha256=start_sha)
    lesson_list.append((lesson_id,prepared_for_lesson))
    file_checks.append((master_file,start_sha))

report_v28=build_pipeline_snapshot(lesson_list,canonical_course=CANONICAL_COURSE,
                                  human_approved=EXPLICIT_APPROVED_TERMS)
for file,sha in file_checks:
    if hashlib.sha256(file.read_bytes()).hexdigest()!=sha:
        raise RuntimeError(f'Source changed during offline analysis: {file}')

base_dir=RUN_DIR/'translation_v2_figma_implementation'
base_dir.mkdir(parents=True,exist_ok=True)
report_tag=fingerprint(report_v28)[:16]
report_path=base_dir/f'generic_course_pipeline_v2_8_{report_tag}.json'
if report_path.exists():
    if read_json(report_path)!=report_v28:raise RuntimeError('Existing report collision. Refusing overwrite')
    print('Reusing identical V2.8 report:',report_path)
else:
    save_new_json(report_path,report_v28)
    print('Saved new offline V2.8 report:',report_path)
print('Lesson count:',report_v28['registry']['lesson_count'])
print('Unit count:',report_v28['registry']['unit_count'])
print('Course identity:',report_v28['registry']['course_identity']['status'])
print('Candidate terms:',report_v28['term_review_queue']['candidate_count'])
print('Approved terms:',len(report_v28['glossary_memory']['terms']))
print('Consistency review flags:',report_v28['terminology_consistency_qa']['pending_review_count'])
print('No translation, WhisperX, model, audio or paid API was executed.')

## 6 — اختر ماذا نفعل بالترجمة الموجودة

- `reuse_existing`: **الموصى به الآن**. يستعمل الـ17 نصاً عربياً الحالية، **صفر API**.
- `review_existing`: مراجعة ثنائية اللغة لوحدات تختارها بواسطة Gemini Free Tier، **طلب واحد لكل وحدة**، دون إعادة الترجمة. ملف QA جديد غير معتمد.
- `regenerate_selected`: إعادة ترجمة وحدات محددة فقط عبر أربع مراحل، **4 طلبات لكل وحدة**؛ لا تستخدمه إلا إذا قررت إعادة صياغة وحدات بعينها.

**لا يمكن ضمان أن حساب Gemini مجاني**؛ إذا كان المشروع مربوطاً بفوترة فقد تنشأ تكلفة. راجع AI Studio قبل السماح بالطلبات. مفتاح `GEMINI_API_KEY` في *Colab Secrets* فقط، ليس في الدفتر أو GitHub. قد تُستخدم بيانات Free Tier لتحسين منتجات Google، فلا ترسل إليها محتوى غير مصرّح.


In [ ]:
from arabic_dubbing_v2.free_backends import text_jobs, GeminiJSONBackend, gemini_client

TRANSLATION_MODE = 'reuse_existing'  # reuse_existing | review_existing | regenerate_selected
SELECTED_TEXT_UNIT_IDS = ['tu_0007']   # تجربة واحدة فقط عند اختيار المراجعة
RUN_TEXT_API = False                   # False = لا اتصالات خارجية
ALLOW_ONLINE_TEXT = False              # إذن مستقل؛ True فقط بعد التحقق من Free Tier
TEXT_MODEL = 'gemini-2.5-flash'        # نموذج له Free Tier حسب صفحة الأسعار؛ قد تختلف الإتاحة
MAX_TEXT_REQUESTS = 1                 # review_existing: 1/وحدة; regenerate_selected: 4/وحدة

# لا نثبت أي مكتبات أو نقرأ أسراراً عندما تكون الاتصالات معطلة.
if RUN_TEXT_API and ALLOW_ONLINE_TEXT:
    import importlib.util
    if (importlib.util.find_spec('google') is None or importlib.util.find_spec('google.genai') is None):
        subprocess.run([sys.executable,'-m','pip','install','-q','-U','google-genai>=2.25.0'],check=True)
    from google.colab import userdata
    key = userdata.get('GEMINI_API_KEY')
    backend = GeminiJSONBackend(gemini_client(key), model=TEXT_MODEL)
else:
    backend = None

text_result = text_jobs(prepared, OUT_DIR/'translation_and_qa_candidates',
                        mode=TRANSLATION_MODE,
                        unit_ids=SELECTED_TEXT_UNIT_IDS,
                        execute=RUN_TEXT_API,
                        allow_online=ALLOW_ONLINE_TEXT,
                        backend=backend,
                        max_requests=MAX_TEXT_REQUESTS)
print(json.dumps(text_result,ensure_ascii=False,indent=2)[:12000])


## 7 — اختر محرك توليد الصوت العربي المجاني للمعاينة

- `plan_only`: **الافتراضي**؛ يطبع الوحدات والنصوص فقط، دون أي توليد أو API.
- `edge_online`: أصوات عربية جاهزة عبر مكتبة Edge TTS واتصال إنترنت خارجي؛ ليست واجهة رسمية مضمونة للإنتاج. جرّب `ar-SA-HamedNeural`.
- `mms_local`: نموذج `facebook/mms-tts-ara` مجاني للتجربة على Colab GPU؛ يتم تنزيل أوزانه أول مرة، ولا يرسل النص إلى TTS API. **ترخيصه CC-BY-NC 4.0: غير تجاري**.
- `gemini_flash_lite`: `gemini-3.8-flash-lite-tts` يدعم العربية وتوجد له Free Tier، لكنه **قد يكلّف إذا كان حسابك مدفوعاً**؛ يحتاج `GEMINI_API_KEY`. خيار صوت مثل `Kore` ويفيد في مقارنة جودة الصوت.

**مهم:** كل ما يُنتج هنا صوت للمراجعة، وليس `final_dub_ar`، ولا مزامنة نهائية. اختر وحدة قصيرة واحدة أولاً، وقارن السمع والمدة قبل تجربة باقي الوحدات.


In [ ]:
from arabic_dubbing_v2.free_backends import preview_jobs

TTS_ENGINE = 'plan_only'     # plan_only | edge_online | mms_local | gemini_flash_lite
PREVIEW_UNIT_IDS = ['tu_0001']  # وحدة واحدة أولاً. لا نولّد 17 وحدة بالخطأ.
RUN_TTS_PREVIEW = False     # False = لا صوت ولا API
ALLOW_ONLINE_TTS = False   # True مطلوب لـ edge_online أو gemini_flash_lite
ALLOW_LOCAL_TTS = False    # True مطلوب لـ mms_local
MAX_TTS_UNITS = 1          # حد صريح يمنع توليد دفعة غير مقصودة
EDGE_VOICE = 'ar-SA-HamedNeural'
EDGE_RATE = '+0%'
GEMINI_VOICE = 'Kore'
GEMINI_TTS_MODEL = 'gemini-3.8-flash-lite-tts'

if RUN_TTS_PREVIEW:
    if TTS_ENGINE == 'edge_online' and ALLOW_ONLINE_TTS:
        subprocess.run([sys.executable,'-m','pip','install','-q','edge-tts'],check=True)
    if TTS_ENGINE == 'mms_local' and ALLOW_LOCAL_TTS:
        subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<5','sentencepiece'],check=True)
    if TTS_ENGINE == 'gemini_flash_lite' and ALLOW_ONLINE_TTS:
        subprocess.run([sys.executable,'-m','pip','install','-q','-U','google-genai>=2.25.0'],check=True)

if RUN_TTS_PREVIEW and TTS_ENGINE == 'gemini_flash_lite' and ALLOW_ONLINE_TTS:
    from google.colab import userdata
    gemini_tts_client = gemini_client(userdata.get('GEMINI_API_KEY'))
else:
    gemini_tts_client = None

preview = preview_jobs(prepared, OUT_DIR/'temporary_audio_previews_v2_6',
                       engine=TTS_ENGINE, unit_ids=PREVIEW_UNIT_IDS,
                       execute=RUN_TTS_PREVIEW, allow_online=ALLOW_ONLINE_TTS,
                       allow_local=ALLOW_LOCAL_TTS,
                       edge_voice=EDGE_VOICE, rate=EDGE_RATE,
                       gemini_client_obj=gemini_tts_client,
                       gemini_voice=GEMINI_VOICE, gemini_model=GEMINI_TTS_MODEL,
                       max_units=MAX_TTS_UNITS)
print(preview['status'])
print(json.dumps(preview.get('plan',preview.get('results')),ensure_ascii=False,indent=2)[:12000])


## 7B — فحص توقيت عينات الصوت السابقة V2.8 (قراءة فقط)

يقرأ سجل WAV السابق إذا كان موجوداً على Google Drive، ويقارن مدده مع توقيتات المصدر ويكشف احتمال تداخل الوحدات. لا يولّد صوتاً ولا يعدل التوقيتات أو WAV.


In [ ]:
# 7B — Read-only audit of the PREVIOUSLY GENERATED audio manifest.
# No voice generation: useful even when Colab GPU is disconnected.
from arabic_dubbing_v2.audio_timeline_v28 import audit_existing_tts_manifest
from arabic_dubbing_v2.workflow import enrich_master
from arabic_dubbing_v2.common import read_json,save_new_json,fingerprint
from pathlib import Path
import hashlib,json

OLD_PILOT_MANIFEST = (RUN_DIR/'gemini_translation_v1'/'editorial_qa_tts_v1'/
    'tts_pilot_3_manifest_qa35lite_20261012T161432Z.json')
if OLD_PILOT_MANIFEST.is_file():
    input_hash=hashlib.sha256(Path(MASTER_PATH).read_bytes()).hexdigest()
    audio_prepared=enrich_master(read_json(MASTER_PATH),
        source_filename=str(MASTER_PATH),source_sha256=input_hash)
    report_audio=audit_existing_tts_manifest(audio_prepared,read_json(OLD_PILOT_MANIFEST))
    if hashlib.sha256(Path(MASTER_PATH).read_bytes()).hexdigest()!=input_hash:
        raise RuntimeError('Master changed: abort audit')
    out=RUN_DIR/'translation_v2_figma_implementation'
    out.mkdir(parents=True,exist_ok=True)
    dest=out/f'audio_timeline_audit_v2_8_{fingerprint(report_audio)[:16]}.json'
    if dest.exists():
        if read_json(dest)!=report_audio:raise RuntimeError('Report collision; refusing overwrite')
    else:save_new_json(dest,report_audio)
    print('Existing audio clips:',report_audio['measured_units'])
    print('Potential overlaps:',report_audio['units_with_potential_next_start_overlap'])
    for record in report_audio['records']:
        print(record['unit_id'],record['source_span_seconds'],'->',
              record['actual_audio_seconds'],'seconds:',record['timing_status'])
    print('Report:',dest)
    print('No TTS calls; no source timing or audio modified.')
else:
    print('Historical audio manifest not found at',OLD_PILOT_MANIFEST)
    print('Nothing generated or changed. Point OLD_PILOT_MANIFEST to an existing JSON to audit it.')

## 8 — خيار منفصل: تجربة Builder عام من RAW WhisperX (لا يغيّر V1.1)

**لا تشغّله على ملف الماستر الذي أُصلحت وحداته بشرياً** إلا إذا أردت تجربة مقارنة مستقلة. يحافظ على كل الكلمات المحاذاة. قد تختلف حدود وحداته عن Builder V1.1، ولذلك لا يستبدل النسخة المراجعة.

In [ ]:
EXPERIMENT_WITH_NEW_BUILDER = False
ALIGNED_RAW_PATH = RUN_DIR / 'whisperx_aligned_raw.json'
if not EXPERIMENT_WITH_NEW_BUILDER:
    print('Experimental Builder disabled; saved reviewed TUs remain unchanged.')
else:
    from arabic_dubbing_v2.boundaries import build_units_from_whisperx
    from arabic_dubbing_v2.common import read_json, save_new_json
    raw=read_json(ALIGNED_RAW_PATH)
    source_segs=raw.get('source_segments',raw.get('segments'))
    comparison=build_units_from_whisperx(source_segs,
                       course=prepared['translation_v2_metadata']['course'])
    dest=OUT_DIR / f'experimental_builder_v2_{PLAN_TAG}.json'
    save_new_json(dest,comparison)
    print('Comparison only:',dest,comparison['builder_report'])


## حالة الإنجاز والخطوة التالية

يمكنك العمل الآن على **النصوص الموجودة** (`reuse_existing`)، أو مراجعتها عبر API اختياري، أو عمل صوت تجريبي **لوحدة واحدة** بمحركات `edge_online` أو `mms_local` أو `gemini_flash_lite`. يحتفظ الدفتر بكل المصادر والتوقيتات والمعرّفات وينشئ ملفات ناتجة منفصلة، ولا يعتمد أي صوت أو ترجمة نهائياً.

- لتجربة الصوت دون مفتاح API: اختر `TTS_ENGINE='mms_local'` و`RUN_TTS_PREVIEW=True` و`ALLOW_LOCAL_TTS=True` (قد تحتاج GPU وتنزيل النموذج)، أو `edge_online` بإذن الاتصال.
- لمراجعة `tu_0007` مع Gemini Free Tier: اختر `TRANSLATION_MODE='review_existing'` و`RUN_TEXT_API=True` و`ALLOW_ONLINE_TEXT=True` بعد إعداد السر والتأكد من فئة الفوترة.
- لا تغيّر `MASTER_FILENAME` للرجوع إلى نسخة قديمة بصمت.
